In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import json

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
import joblib

In [ ]:
URL = "http://localhost:8080/api/ml/dataset"

response = requests.get(URL)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

print("Real data shape:", df.shape)

Loading mock data and combining with real data

In [ ]:
with open("../data/mock_data.json", "r") as f:
    mock = json.load(f)

df_mock = pd.DataFrame(mock)

df_combined = pd.concat([df, df_mock], ignore_index=True)

print("Real rows:", len(df))
print("mock rows:", len(df_mock))
print("Combined rows:", len(df_combined))
print("\nLiked distribution:\n", df_combined["liked"].value_counts())


Cleaning the data:
   - text_columns --> contain the names of all the columns that contain words
   - fillna --> to fill N/A into cells that are empty, intead of leaving a blank gap
   - astype(str) --> make sure every value is treated as a string
   - df["liked"].astype(int) --> liked column marks wether user likes an artwork or not 
      - (1 liked, 0 not liked)
      
combining the columns:
   - to have the describtive features in one line rather than multipe columns, makes it easier to associate each feature with a specific artwork.

In [ ]:

text_columns = [
    "artist", "period", "culture", "medium",
    "preferredArtists", "preferredStyles",
    "preferredMediums", "preferredTimePeriods",
    "preferredMovements"
]

for col in text_columns:
    df_combined[col] = df_combined[col].fillna("").astype(str)

df_combined["liked"] = df_combined["liked"].astype(int)


# Combined Text Feature
df_combined["combined_text"] = (
    df_combined["artist"] + " " +
    df_combined["period"] + " " +
    df_combined["culture"] + " " +
    df_combined["medium"] + " " +
    df_combined["preferredArtists"] + " " +
    df_combined["preferredStyles"] + " " +
    df_combined["preferredMediums"] + " " +
    df_combined["preferredTimePeriods"] + " " +
    df_combined["preferredMovements"]
)

X_text = df_combined["combined_text"]
y = df_combined["liked"]

print("Total rows:", len(df_combined))
print("\nLiked distribution:\n", df_combined["liked"].value_counts())

# bar chart of liked distribution
df_combined["liked"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["Not Liked (0)", "Liked (1)"], rotation=0)
plt.ylabel("Count")
plt.title("Liked/Not Liked Distribution")
plt.tight_layout()
plt.show()

splitting data 
- training data 80%
- testing data 20% 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


TF-IDF Vectorization - turning words into numbers 
- max_features=500 keeps the 500 most informative words
- ngram_range=(1,2) captures single words AND two-word pairs like "oil painting"

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## Trainning models

Logistic regression:
- checks the features and finds patterns
- finds a single global boundary separating liked from not liked
- learns a weight for each of the 500 TF-IDF features

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

#testing the trained model
y_pred = model.predict(X_test_tfidf)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# confusion matrix
cm_lr = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=["Not Liked", "Liked"])
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

- artwork_recommender :
    - trained model
- vectorizer: 
    - word to number translator 

In [ ]:
joblib.dump(model, "artwork_recommender.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")